# Training in colab

This is a notebook to train the transformer model in google colab. Colab's free gpu tier is not sufficent (~5hr usage, slow gpu, slow cpu) for training. Therefore you need compute credits which can either be purchased (quite expensive - not recommended) or are already included in some subscriptions (maybe you already have some). 

### Which runtime type should I choose?
Training works really well on a Nvidia G4 (same as the RTX 6000 Blackwell; ~10 credits/hr) taking around 8hr. I haven't trained on TPUs but it might be an interesting experiment so if you try, please share the results :)

### Datasets
Converting datasets takes a lot of time, so it makes sense to convert the datasets on your on device. Depending on your internet connection speed it may be good to first upload the files to your gdrive since the download speed between drive and colab env is high (~40MB/s). 

In [ ]:
# MOUNT GDRIVE FOR DATASET DOWNLOAD
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# INSTALL HOMR
!pip install -q uv

# if you're training on your fork replace this with it
!git clone https://github.com/liebharc/homr.git
%cd /content/homr

# switch to your branch
!git switch feature/my_branch
!mkdir -p datasets

# onnx-simplifier is rebuild every time which makes this step take
# really long, you should therefore remove it from pyproject.toml
!uv pip install --system -r pyproject.toml --group dev --extra cuda

In [ ]:
# COPY DATASETS TO DISK
from os import makedirs
import shutil

source = '/content/drive/MyDrive/datasets' # may be a different location for you
destination = '/content/homr/datasets/'

shutil.copytree(source, destination, dirs_exist_ok=True)

In [ ]:
# EXTRACT DATASETS
!7z x /content/homr/datasets/musetrainer.7z -o/content/homr/datasets/musetrainer
!7z x /content/homr/datasets/lieder.7z -o/content/homr/datasets/Lieder-main
!7z x /content/homr/datasets/pdmx.7z -o/content/homr/datasets/pdmx
!7z x /content/homr/datasets/grandstaff_data.7z -o/content/homr/datasets/grandstaff
!7z x /content/homr/datasets/corpus_data.7z -o/content/homr/datasets/Corpus

In [ ]:
# CHECKS
!nvcc --version # cuda version
!nproc # number threads for dataloader_num_workers

In [ ]:
# TRAIN TRANSFORMER
# Make sure you set batch_size, gradient_accumulation_steps and dataloader_num_workers correctly
# Also create frequent backups of the current_training folder so you don't loose progress
# when you get disconnected from the gpu (I did some research on this and it seems like it can 
# happen even if you're on paid credits)
!python3 -m training.train transformer